# 05 — Figuras dos cenários sintéticos

Gera, sem reexecutar os otimizadores, superfícies e mapas de contorno dos RSMs ajustados para a semente representativa 101, confrontados com as respostas verdadeiras, e visualizações da fronteira de Pareto verdadeira amostrada diretamente do casco convexo das âncoras.

Como a fronteira tem 4, 6 ou 12 objetivos, sua representação completa exige projeções: o notebook exporta o conjunto de Pareto em espaço de decisão, PCA e coordenadas paralelas, além de todas as projeções bidimensionais entre pares de objetivos.

In [ ]:
from pathlib import Path
from itertools import combinations, product
import json, math, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import colors
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D

def project_root(start=Path.cwd()):
    p=start.resolve()
    for candidate in (p,*p.parents):
        if (candidate/'configs'/'smoke.json').exists(): return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada')

ROOT=project_root(); MODE=os.environ.get('CNBI_MODE','SMOKE').upper()
CFG=json.loads((ROOT/'configs'/f'{MODE.lower()}.json').read_text(encoding='utf-8'))
ALPHA=2**0.75; SURFACE_SEED=101; GRID_SIZE=81
FIG_ROOT=ROOT/'results'/'synthetic'/'figures'; SURFACE_DIR=FIG_ROOT/'response_surfaces'; PARETO_DIR=FIG_ROOT/'true_pareto'
SURFACE_DIR.mkdir(parents=True,exist_ok=True); PARETO_DIR.mkdir(parents=True,exist_ok=True)
LEVEL_LABEL={'low':'baixa','medium':'média','high':'alta'}
plt.rcParams.update({'font.family':'DejaVu Serif','font.size':9,'axes.titlesize':10,'axes.labelsize':9,'legend.fontsize':8,'figure.titlesize':13,'savefig.facecolor':'white','axes.facecolor':'white'})


In [ ]:
# Superfícies: RSM ajustado (cor) e função verdadeira (malha/isolinha).
def design_matrix(X):
    X=np.asarray(X,float); x1,x2,x3=X.T
    return np.column_stack([np.ones(len(X)),x1,x2,x3,x1*x1,x2*x2,x3*x3,x1*x2,x1*x3,x2*x3])

def fitted_rsm(anchors,seed=SURFACE_SEED):
    factorial=np.array(list(product([-1.,1.],repeat=3)),float)
    axial=np.vstack([np.eye(3)*ALPHA,-np.eye(3)*ALPHA])
    X=np.vstack([factorial,axial,np.zeros((5,3))]); D=design_matrix(X)
    F=np.sum((X[:,None,:]-anchors[None,:,:])**2,axis=2)
    rng=np.random.default_rng(seed); sigma=np.sqrt(F.var(0,ddof=1)*(.05/.95))
    Y=F+rng.normal(0,sigma,F.shape)
    return np.linalg.lstsq(D,Y,rcond=None)[0]

def response_slices(B,anchor,objective):
    axis=np.linspace(-ALPHA,ALPHA,GRID_SIZE); U,V=np.meshgrid(axis,axis)
    slices=[]
    for a,b,fixed in ((0,1,2),(0,2,1),(1,2,0)):
        pts=np.zeros((U.size,3)); pts[:,a]=U.ravel(); pts[:,b]=V.ravel()
        mask=(pts*pts).sum(1)<=ALPHA**2+1e-12
        fit=(design_matrix(pts)@B[:,objective]).reshape(U.shape)
        truth=np.sum((pts-anchor[None,:])**2,axis=1).reshape(U.shape)
        fit[~mask.reshape(U.shape)]=np.nan; truth[~mask.reshape(U.shape)]=np.nan
        slices.append((a,b,fixed,U,V,fit,truth))
    return slices

def response_figure(scenario,objective,anchors,B):
    anchor=anchors[objective]; slices=response_slices(B,anchor,objective)
    values=np.concatenate([z[np.isfinite(z)] for item in slices for z in item[-2:]])
    vmin,vmax=float(values.min()),float(values.max()); norm=colors.Normalize(vmin=vmin,vmax=vmax); cmap='viridis'
    fig=plt.figure(figsize=(15.5,9.2),layout='constrained'); axes=[]
    for col,(a,b,fixed,U,V,fit,truth) in enumerate(slices):
        ax=fig.add_subplot(2,3,col+1,projection='3d'); axes.append(ax)
        ax.plot_surface(U,V,fit,cmap=cmap,norm=norm,rstride=2,cstride=2,linewidth=0,antialiased=True,alpha=.92)
        ax.plot_wireframe(U[::6,::6],V[::6,::6],truth[::6,::6],color='.12',linewidth=.45,alpha=.55)
        ax.set(xlabel=rf'$x_{{{a+1}}}$',ylabel=rf'$x_{{{b+1}}}$',zlabel=rf'$f_{{{objective+1}}}$')
        ax.set_title(rf'Superfície: $x_{{{a+1}}}\times x_{{{b+1}}}$; $x_{{{fixed+1}}}=0$')
        ax.view_init(elev=28,azim=-132)
        ax2=fig.add_subplot(2,3,col+4); axes.append(ax2)
        levels=np.linspace(vmin,vmax,15); ax2.contourf(U,V,fit,levels=levels,cmap=cmap,norm=norm)
        ax2.contour(U,V,truth,levels=levels[1:-1:2],colors='.12',linewidths=.55,alpha=.75)
        ax2.scatter(anchor[a],anchor[b],marker='*',s=90,c='white',edgecolors='.1',linewidths=.8,zorder=5)
        circle=plt.Circle((0,0),ALPHA,fill=False,color='.25',linewidth=.7); ax2.add_patch(circle)
        ax2.set_aspect('equal'); ax2.set(xlabel=rf'$x_{{{a+1}}}$',ylabel=rf'$x_{{{b+1}}}$')
        ax2.set_title(rf'Contorno: $x_{{{a+1}}}\times x_{{{b+1}}}$; $x_{{{fixed+1}}}=0$')
    level=scenario.split('_',1)[1]
    fig.suptitle(rf'{scenario}: resposta $f_{{{objective+1}}}$ — RSM semente {SURFACE_SEED} versus função verdadeira')
    legend=[Line2D([0],[0],color='#440154',lw=5,label='RSM ajustado'),Line2D([0],[0],color='.12',lw=.8,label='função verdadeira'),Line2D([0],[0],marker='*',color='none',markerfacecolor='white',markeredgecolor='.1',markersize=9,label='ótimo projetado')]
    fig.legend(handles=legend,loc='outside lower center',ncol=3,frameon=False)
    sm=plt.cm.ScalarMappable(norm=norm,cmap=cmap); sm.set_array([]); fig.colorbar(sm,ax=axes,shrink=.72,pad=.02,label=rf'resposta $f_{{{objective+1}}}$')
    return fig


In [ ]:
# Fronteiras verdadeiras: nenhuma solução de método é usada nestas figuras.
PARETO_FONT_SIZE=14; PARETO_TITLE_SIZE=17; PARETO_LABEL_SIZE=16; PARETO_TICK_SIZE=14; PARETO_LEGEND_SIZE=14; PARETO_STAR_SIZE=130

def style_pareto_axis(ax):
    ax.title.set_fontsize(PARETO_TITLE_SIZE)
    ax.xaxis.label.set_size(PARETO_LABEL_SIZE); ax.yaxis.label.set_size(PARETO_LABEL_SIZE)
    ax.tick_params(axis='both',labelsize=PARETO_TICK_SIZE)
    if hasattr(ax,'zaxis'):
        ax.zaxis.label.set_size(PARETO_LABEL_SIZE); ax.zaxis.set_tick_params(labelsize=PARETO_TICK_SIZE)

def pareto_legend_handles():
    return [
        Line2D([0],[0],marker='o',linestyle='none',markerfacecolor='#31688e',markeredgecolor='none',markersize=8,label='pontos da fronteira de Pareto verdadeira'),
        Line2D([0],[0],marker='*',linestyle='none',markerfacecolor='crimson',markeredgecolor='.15',markersize=15,label='ótimos individuais verdadeiros'),
    ]

def sample_indices(n,limit,seed):
    if n<=limit: return np.arange(n)
    return np.sort(np.random.default_rng(seed).choice(n,size=limit,replace=False))

def pca_projection(Fn):
    centered=Fn-Fn.mean(0); _,s,vt=np.linalg.svd(centered,full_matrices=False)
    score=centered@vt[:3].T; ratio=s*s/np.sum(s*s)
    return score,ratio[:3]

def pareto_overview(scenario,anchors,Xref,Fref,ideal,nadir):
    idx=sample_indices(len(Xref),6000,20260821+len(anchors)); X=Xref[idx]; F=Fref[idx]
    amp=np.maximum(nadir-ideal,1e-12); Fn=(F-ideal)/amp; scores,ratio=pca_projection(Fn); color=Fn.mean(1)
    fig=plt.figure(figsize=(16,5.4),layout='constrained')
    ax_dec=fig.add_subplot(1,3,1,projection='3d'); sc=ax_dec.scatter(X[:,0],X[:,1],X[:,2],c=color,cmap='viridis',s=3,alpha=.28,rasterized=True)
    ax_dec.scatter(anchors[:,0],anchors[:,1],anchors[:,2],marker='*',s=PARETO_STAR_SIZE,c='crimson',edgecolors='.15',linewidths=.55)
    ax_dec.set(xlabel='$x_1$',ylabel='$x_2$',zlabel='$x_3$',title=r'Conjunto de Pareto verdadeiro $\mathcal{P}_x$'); style_pareto_axis(ax_dec)
    ax_dec.legend(handles=pareto_legend_handles(),frameon=False,loc='upper left',fontsize=PARETO_LEGEND_SIZE)
    ax_pca=fig.add_subplot(1,3,2,projection='3d'); ax_pca.scatter(scores[:,0],scores[:,1],scores[:,2],c=color,cmap='viridis',s=3,alpha=.28,rasterized=True)
    ax_pca.set(xlabel=f'CP1 ({100*ratio[0]:.1f}%)',ylabel=f'CP2 ({100*ratio[1]:.1f}%)',zlabel=f'CP3 ({100*ratio[2]:.1f}%)',title='Fronteira objetiva normalizada — PCA'); style_pareto_axis(ax_pca)
    ax=fig.add_subplot(1,3,3); par=sample_indices(len(Fn),450,777+len(anchors)); xx=np.arange(1,Fn.shape[1]+1)
    for row in Fn[par]: ax.plot(xx,row,color='#31688e',alpha=.055,linewidth=.55)
    Fanchors=np.sum((anchors[:,None,:]-anchors[None,:,:])**2,axis=2); Fanchors=(Fanchors-ideal)/amp
    for row in Fanchors: ax.plot(xx,row,color='crimson',alpha=.75,linewidth=.8)
    ax.set_xticks(xx); ax.set_xticklabels([rf'$f_{{{j}}}$' for j in xx]); ax.set(xlabel='objetivo',ylabel='resposta normalizada',title='Coordenadas paralelas'); ax.grid(axis='y',alpha=.2); style_pareto_axis(ax)
    cbar=fig.colorbar(sc,ax=[ax_dec,ax_pca],orientation='horizontal',shrink=.52,pad=.08)
    cbar.set_label('média das respostas normalizadas',fontsize=PARETO_LABEL_SIZE); cbar.ax.tick_params(labelsize=PARETO_TICK_SIZE)
    return fig

def pareto_pairwise_pages(scenario,Fref,anchors,pdf_path,page_dir):
    pairs=list(combinations(range(Fref.shape[1]),2)); idx=sample_indices(len(Fref),5000,4242+Fref.shape[1]); F=Fref[idx]
    Fanchors=np.sum((anchors[:,None,:]-anchors[None,:,:])**2,axis=2); records=[]; page_dir.mkdir(parents=True,exist_ok=True)
    with PdfPages(pdf_path) as pdf:
        for page,start in enumerate(range(0,len(pairs),6),1):
            fig,axs=plt.subplots(2,3,figsize=(15.8,9.6),layout='constrained'); subset=pairs[start:start+6]
            for ax in axs.ravel(): ax.set_visible(False)
            page_records=[]
            for ax,(i,j) in zip(axs.ravel(),subset):
                ax.set_visible(True); ax.scatter(F[:,i],F[:,j],s=3,c='#31688e',alpha=.22,linewidths=0,rasterized=True)
                ax.scatter(Fanchors[:,i],Fanchors[:,j],marker='*',s=PARETO_STAR_SIZE,c='crimson',edgecolors='.15',linewidths=.55,zorder=3)
                ax.set(xlabel=rf'$f_{{{i+1}}}$',ylabel=rf'$f_{{{j+1}}}$',title=rf'Projeção $f_{{{i+1}}}\times f_{{{j+1}}}$'); ax.grid(alpha=.15); style_pareto_axis(ax)
                page_records.append({'scenario':scenario,'page':page,'objective_i':i+1,'objective_j':j+1})
            fig.legend(handles=pareto_legend_handles(),loc='outside lower center',ncol=2,frameon=False,fontsize=PARETO_LEGEND_SIZE)
            pdf.savefig(fig,dpi=300,bbox_inches='tight'); png=page_dir/f'{scenario}_pareto_pairwise_page{page:02d}.png'; fig.savefig(png,dpi=240,bbox_inches='tight'); plt.close(fig)
            for row in page_records: row['page_png']=png.relative_to(ROOT).as_posix()
            records.extend(page_records)
    return records


In [ ]:
# Execução idempotente somente sobre artefatos científicos existentes.
scenarios=[f'm{m}_{level}' for m in CFG['scenario_objectives'] for level in CFG['correlation_targets']]
surface_manifest=[]; pareto_manifest=[]
for scenario in scenarios:
    scenario_data=np.load(ROOT/'data'/'generated'/f'{scenario}_scenario.npz'); anchors=np.asarray(scenario_data['anchors'],float); B=fitted_rsm(anchors)
    scenario_surface=SURFACE_DIR/scenario; scenario_surface.mkdir(parents=True,exist_ok=True); atlas=scenario_surface/f'{scenario}_rsm_seed{SURFACE_SEED}_atlas.pdf'
    with PdfPages(atlas) as pdf:
        for objective in range(len(anchors)):
            fig=response_figure(scenario,objective,anchors,B); png=scenario_surface/f'{scenario}_f{objective+1:02d}_rsm_seed{SURFACE_SEED}.png'
            fig.savefig(png,dpi=240,bbox_inches='tight'); pdf.savefig(fig,dpi=300,bbox_inches='tight'); plt.close(fig)
            surface_manifest.append({'mode':MODE,'scenario':scenario,'objective':objective+1,'rsm_seed':SURFACE_SEED,'slice_fixed_value':0.0,'png':png.relative_to(ROOT).as_posix(),'atlas_pdf':atlas.relative_to(ROOT).as_posix()})
    reference=np.load(ROOT/'data'/'reference_fronts'/f'{scenario}_pareto_reference.npz'); Xref=np.asarray(reference['X'],float); Fref=np.asarray(reference['F'],float); ideal=np.asarray(reference['ideal_true'],float); nadir=np.asarray(reference['nadir_true'],float)
    overview=pareto_overview(scenario,anchors,Xref,Fref,ideal,nadir); overview_png=PARETO_DIR/f'{scenario}_true_pareto_overview.png'; overview_pdf=PARETO_DIR/f'{scenario}_true_pareto_overview.pdf'
    overview.savefig(overview_png,dpi=240,bbox_inches='tight'); overview.savefig(overview_pdf,dpi=300,bbox_inches='tight'); plt.close(overview)
    pair_pdf=PARETO_DIR/f'{scenario}_true_pareto_pairwise.pdf'; pair_dir=PARETO_DIR/f'{scenario}_pairwise_pages'; pairs=pareto_pairwise_pages(scenario,Fref,anchors,pair_pdf,pair_dir)
    for row in pairs: row.update({'mode':MODE,'overview_png':overview_png.relative_to(ROOT).as_posix(),'overview_pdf':overview_pdf.relative_to(ROOT).as_posix(),'pairwise_pdf':pair_pdf.relative_to(ROOT).as_posix()})
    pareto_manifest.extend(pairs); scenario_data.close(); reference.close()

surface_table=pd.DataFrame(surface_manifest); pareto_table=pd.DataFrame(pareto_manifest)
surface_table.to_csv(FIG_ROOT/f'{MODE.lower()}_response_surface_manifest.csv',index=False); pareto_table.to_csv(FIG_ROOT/f'{MODE.lower()}_true_pareto_manifest.csv',index=False)
metadata={'mode':MODE,'response_surface_kind':'quadratic RSM fitted from the 19-run CCD with independent Gaussian noise','response_surface_seed':SURFACE_SEED,'ground_truth_overlay':True,'slice_fixed_value':0.0,'grid_size':GRID_SIZE,'pareto_source':'direct sample from conv(anchors)','pareto_reference_points':int(CFG['reference_points']),'objective_projection':'all pairwise projections; PCA and parallel coordinates are normalized only for display'}
(FIG_ROOT/f'{MODE.lower()}_figure_metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
print(f'Figuras {MODE}: {len(surface_table)} respostas, {len(pareto_table)} projeções de Pareto, {len(scenarios)} cenários.')
